# Aula 5 — Validação de Schema e Contratos de Dados

Este notebook acompanha o hands-on da Aula 5 da disciplina **Validação de Dados em ML e Bibliotecas de Qualidade (POSTECH/FIAP)**.

**Ferramentas:** Pandera, Pydantic v2, JSON Schema (`jsonschema`)

**O que você vai ver:**
1. Schema como código com **Pandera DataFrameModel** (class-based)
2. Validação cross-field com **Pydantic v2 model_validator**
3. Reutilização de definições em **JSON Schema** com `$ref` e `$defs`

---
## 0. Instalação e Importações

In [ ]:
# Importações gerais
import pandas as pd
import numpy as np

---
## 1. Pandera — Schema como Código com DataFrameModel

O **Pandera** permite definir schemas como classes Python, usando type hints e `pa.Field` para as restrições.

Vantagens do `DataFrameModel` (class-based) em relação ao `DataFrameSchema` (dict-based):
- Sintaxe familiar de dataclasses Python
- Suporte nativo a type hints e type checkers (mypy, pyright)
- Pode ser usado como decorator `@pa.check_types` em funções
- Melhor integração com CI/CD e versionamento

**Referência:** [Pandera DataFrameModel Docs](https://pandera.readthedocs.io/en/stable/dataframe_models.html)

In [2]:
import pandera as pa
from pandera.typing import Series, DataFrame

# Schema como classe Python — versionável e testável em CI/CD
class TransactionSchema(pa.DataFrameModel):
    transaction_id: Series[str] = pa.Field(nullable=False, unique=True)
    customer_id:    Series[str] = pa.Field(nullable=False)
    amount:         Series[float] = pa.Field(gt=0, le=1_000_000)
    category:       Series[str] = pa.Field(isin=["food", "transport",
                                                  "retail", "other"])
    is_fraud:       Series[bool] = pa.Field(nullable=True)

    class Config:
        strict = True   # rejeita colunas inesperadas
        coerce = True   # converte tipos automaticamente

print("Schema definido:", TransactionSchema.__name__)

Schema definido: TransactionSchema


/Users/luiz.braz/Personal-Labs/validacao-de-dados-blibliotecas-qualidade/.venv/lib/python3.13/site-packages/pandera/_pandas_deprecated.py:154: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


In [3]:
# --- Caso válido ---
df_valido = pd.DataFrame({
    "transaction_id": ["TX0000000001", "TX0000000002"],
    "customer_id":    ["C0001", "C0002"],
    "amount":         [100.0, 250.50],
    "category":       ["food", "transport"],
    "is_fraud":       [False, None],
})

validated = TransactionSchema.validate(df_valido)
print("DataFrame válido:")
print(validated)

DataFrame válido:
  transaction_id customer_id  amount   category  is_fraud
0   TX0000000001       C0001   100.0       food     False
1   TX0000000002       C0002   250.5  transport     False


In [4]:
# --- Caso inválido: múltiplas violações detectadas de uma vez (lazy=True) ---
df_invalido = pd.DataFrame({
    "transaction_id": ["TX001", "TX001"],  # ID duplicado
    "customer_id":    ["C0001", "C0002"],
    "amount":         [100.0, -50.0],       # valor negativo
    "category":       ["food", "casino"],   # categoria inválida
    "is_fraud":       [False, None],
})

try:
    TransactionSchema.validate(df_invalido, lazy=True)
except pa.errors.SchemaErrors as e:
    print("Violações encontradas:")
    print(e.failure_cases[["column", "failure_case", "check"]].to_string(index=False))

Violações encontradas:
        column failure_case                                          check
transaction_id        TX001                               field_uniqueness
transaction_id        TX001                               field_uniqueness
        amount        -50.0                                greater_than(0)
      category       casino isin(['food', 'transport', 'retail', 'other'])


In [5]:
# --- DataFrameModel como decorator: garante contrato na entrada e saída ---
@pa.check_types
def preprocessar(df: DataFrame[TransactionSchema]) -> DataFrame[TransactionSchema]:
    """Arredonda o amount para 2 casas decimais."""
    df = df.copy()
    df["amount"] = df["amount"].round(2)
    return df

resultado = preprocessar(df_valido)
print("DataFrame pós-processamento:")
print(resultado)

DataFrame pós-processamento:
  transaction_id customer_id  amount   category  is_fraud
0   TX0000000001       C0001   100.0       food     False
1   TX0000000002       C0002   250.5  transport     False


---
## 2. Pydantic v2 — Validação Cross-field com `model_validator`

O **Pydantic v2** é a biblioteca de validação mais popular do ecossistema Python (~200M downloads/mês). A versão 2 reescreveu o core em Rust, resultando em ganho de performance de 5x a 50x.

O `@model_validator(mode="after")` permite validar **regras de negócio que envolvem múltiplos campos** — algo impossível com `@field_validator`, que valida um campo de forma isolada.

**Referência:** [Pydantic model_validator Docs](https://docs.pydantic.dev/latest/concepts/validators/#model-validators)

In [6]:
from pydantic import BaseModel, Field, model_validator
from typing import Optional
from enum import Enum

class Category(str, Enum):
    FOOD = "food"
    TRANSPORT = "transport"
    RETAIL = "retail"
    OTHER = "other"

class Transaction(BaseModel):
    transaction_id: str = Field(..., min_length=10)
    customer_id:    str = Field(..., min_length=5)
    amount:         float = Field(..., gt=0, le=1_000_000)
    category:       Category
    is_online:      bool = False
    is_fraud:       Optional[bool] = None

    # Validação cross-field: regra de negócio entre múltiplos campos
    @model_validator(mode="after")
    def online_transaction_limit(self):
        if self.is_online and self.amount > 10_000:
            raise ValueError(
                f"Transação online acima de R$10.000 requer revisão manual "
                f"(valor: R${self.amount:.2f})"
            )
        return self

print("Modelo Transaction definido.")

Modelo Transaction definido.


In [7]:
# --- Caso válido ---
t = Transaction(
    transaction_id="TX0000000001",
    customer_id="C0001",
    amount=500.0,
    category="food",
    is_online=True
)
print(f"Válido: {t.transaction_id} — R${t.amount:.2f} (online={t.is_online})")

Válido: TX0000000001 — R$500.00 (online=True)


In [8]:
# --- Caso inválido: viola regra cross-field (online + amount > 10k) ---
try:
    Transaction(
        transaction_id="TX0000000002",
        customer_id="C0002",
        amount=15_000.0,
        category="retail",
        is_online=True
    )
except Exception as e:
    print(f"Erro de validação cross-field:")
    for err in e.errors():
        print(f"  [{err['loc']}] {err['msg']}")

Erro de validação cross-field:
  [()] Value error, Transação online acima de R$10.000 requer revisão manual (valor: R$15000.00)


In [9]:
# --- Comparação: mesmo valor alto, mas transação presencial (is_online=False) ---
t_presencial = Transaction(
    transaction_id="TX0000000003",
    customer_id="C0003",
    amount=15_000.0,
    category="retail",
    is_online=False  # presencial: regra não se aplica
)
print(f"Válido (presencial): R${t_presencial.amount:.2f}")

Válido (presencial): R$15000.00


---
## 3. JSON Schema — Reutilização com `$defs` e `$ref`

O **JSON Schema** é um padrão de linguagem neutra (30+ implementações). Diferente de Pandera e Pydantic, ele é ideal para:
- Contratos entre serviços em diferentes linguagens
- Validação de eventos em plataformas de streaming (Kafka, etc.)
- Documentação de APIs

O mecanismo `$defs` + `$ref` permite **definir tipos reutilizáveis** dentro do schema — evitando duplicação de regras e garantindo consistência.

**Referência:** [JSON Schema Draft 2020-12](https://json-schema.org/draft/2020-12)

In [10]:
import jsonschema, json

# Schema com reutilização via $defs — evita repetição de regras
schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$defs": {
        "Amount": {  # definição reutilizável em múltiplos campos
            "type": "number",
            "exclusiveMinimum": 0,
            "maximum": 1000000,
            "description": "Valor monetário positivo em reais"
        }
    },
    "title": "Transaction",
    "type": "object",
    "required": ["transaction_id", "customer_id", "amount", "category"],
    "properties": {
        "transaction_id": {"type": "string", "minLength": 10},
        "customer_id":    {"type": "string"},
        "amount":         {"$ref": "#/$defs/Amount"},  # reutiliza a definição
        "category":       {"type": "string",
                           "enum": ["food", "transport", "retail", "other"]},
        "is_fraud":       {"type": ["boolean", "null"]}
    },
    "additionalProperties": False
}

print("Schema definido com $defs/Amount reutilizável")
print(json.dumps(schema["$defs"], indent=2))

Schema definido com $defs/Amount reutilizável
{
  "Amount": {
    "type": "number",
    "exclusiveMinimum": 0,
    "maximum": 1000000,
    "description": "Valor monet\u00e1rio positivo em reais"
  }
}


In [11]:
# --- Caso válido ---
payload_valido = {
    "transaction_id": "TX0000000001",
    "customer_id":    "C0001",
    "amount":         125.50,
    "category":       "food"
}

jsonschema.validate(instance=payload_valido, schema=schema)
print("Payload válido:", json.dumps(payload_valido))

Payload válido: {"transaction_id": "TX0000000001", "customer_id": "C0001", "amount": 125.5, "category": "food"}


In [12]:
# --- Casos inválidos ---
casos_invalidos = [
    {
        "descricao": "amount negativo",
        "payload": {"transaction_id": "TX0000000002", "customer_id": "C0002",
                    "amount": -100.0, "category": "food"}
    },
    {
        "descricao": "category inválida",
        "payload": {"transaction_id": "TX0000000003", "customer_id": "C0003",
                    "amount": 50.0, "category": "casino"}
    },
    {
        "descricao": "campo extra não permitido",
        "payload": {"transaction_id": "TX0000000004", "customer_id": "C0004",
                    "amount": 50.0, "category": "food", "campo_extra": "valor"}
    },
]

for caso in casos_invalidos:
    try:
        jsonschema.validate(instance=caso["payload"], schema=schema)
    except jsonschema.ValidationError as e:
        print(f"  [{caso['descricao']}] Erro: {e.message}")

  [amount negativo] Erro: -100.0 is less than or equal to the minimum of 0
  [category inválida] Erro: 'casino' is not one of ['food', 'transport', 'retail', 'other']
  [campo extra não permitido] Erro: Additional properties are not allowed ('campo_extra' was unexpected)


---
## 4. Resumo: Quando usar cada ferramenta

| Ferramenta | Use quando... |
|---|---|
| **Pandera** | Validar DataFrames Pandas/Polars em pipelines de ML |
| **Pydantic v2** | Validar entidades individuais (APIs, configs, payloads) |
| **JSON Schema** | Contratos entre serviços em múltiplas linguagens |
| **TFDV** | Grandes datasets e monitoramento de training-serving skew |
| **Data Contract Spec** | Formalizar SLAs organizacionais entre equipes |

> **Dica:** Comece com Pandera + Pydantic. Avance para JSON Schema quando precisar de interoperabilidade. Use Data Contracts quando precisar de governança organizacional.